# BaseLine

In [1]:
from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator

In [2]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
# Keep only the columns we need for 2-period CUPAC
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),
    "y": TargetRole(cofounders=["X1", "X2"]),

    "y_lag1": PreTargetRole(parent="y", lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

# Multilevel CUPAC with linear regression
test_cupac_linear = ABTest(
    enable_cupac=True,
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,y,1,0.948518,1.590543,0.642025,67.687127,NOT OK,0.064382
1,y_cupac,1,1.007879,1.478836,0.470957,46.727496,OK,0.026723


# Save and Load

## Save

In [3]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()

df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1', 'y'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),

    "y_lag1": PreTargetRole(parent="y", cofounders=["X1", "X2"], lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

test_cupac_linear = ABTest(
    enable_cupac=True,
    save_experiment=True
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

""


In [4]:
result_cupac_linear.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real,control_mean_bias,test_mean_bias
0,y,ridge,66.0358,None,None,None


## Load

In [5]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
# Keep only the columns we need for 2-period CUPAC
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),
    "y": TargetRole(cofounders=["X1", "X2"]),

    "y_lag1": PreTargetRole(parent="y", lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

import os
exp_id = os.listdir(".hypex_experiments")[0] 

test_cupac_linear = ABTest(
    enable_cupac=True,
    load_experiment=exp_id,
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,y,1,0.948518,1.590543,0.642025,67.687127,NOT OK,0.064382
1,y_cupac,1,1.007879,1.478836,0.470957,46.727496,OK,0.026723


In [6]:
result_cupac_linear.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real,control_mean_bias,test_mean_bias
0,y,ridge,66.0358,62.473053,-0.059361,0.111708


# Test CV Folds (без numpy/sklearn)

In [7]:
# Reload modules to get updated Splitter
import sys
if 'hypex' in sys.modules:
    modules_to_remove = [key for key in sys.modules.keys() if key.startswith('hypex')]
    for module in modules_to_remove:
        del sys.modules[module]

from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator
from hypex.splitters import CUPACDataSplitter
from hypex.dataset import ExperimentData

print("✓ Modules reloaded")

# Generate test data
gen = DataGenerator(
    n_samples=100,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
        "d": TreatmentRole(),
        "y": TargetRole(cofounders=["X1", "X2"]),
        "y_lag1": PreTargetRole(parent="y", lag=1),
        "X1_lag1": FeatureRole(parent="X1", lag=1),
        "X2_lag1": FeatureRole(parent="X2", lag=1),
        "y_lag2": PreTargetRole(parent="y", lag=2),
        "X1_lag2": FeatureRole(parent="X1", lag=2),
        "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

exp_data = ExperimentData(data=data)

# Test with CV folds (3 folds)
print("\n=== Test: CUPACDataSplitter with 3 CV folds ===")
splitter = CUPACDataSplitter(n_folds=3, random_state=42, generate_cv_folds=True)
ml_data = splitter.execute(exp_data)
ml_obj = ml_data.get_ml_data('y')

print(f"\nMLData: {ml_obj}")
print(f"Number of CV folds: {len(ml_obj.crossval)}")

# Check fold sizes
total_val_samples = 0
for fold_id, (X_val, Y_val) in ml_obj.crossval.items():
    print(f"\nFold {fold_id}:")
    print(f"  X_val shape: {X_val.shape}")
    print(f"  Y_val shape: {Y_val.shape}")
    total_val_samples += X_val.shape[0]

print(f"\nTotal validation samples across folds: {total_val_samples}")
print(f"Total training samples: {ml_obj.X_train.shape[0]}")
print(f"Match: {total_val_samples == ml_obj.X_train.shape[0]}")

print("\n✓ Test passed! CV folds работают без numpy/sklearn")

✓ Modules reloaded

=== Test: CUPACDataSplitter with 3 CV folds ===

MLData: MLData(X_train=(100, 3), Y_train=(100, 1), X_predict=(100, 3), n_folds=3)
Number of CV folds: 3

Fold 0:
  X_val shape: (34, 3)
  Y_val shape: (34, 1)

Fold 1:
  X_val shape: (33, 3)
  Y_val shape: (33, 1)

Fold 2:
  X_val shape: (33, 3)
  Y_val shape: (33, 1)

Total validation samples across folds: 100
Total training samples: 100
Match: True

✓ Test passed! CV folds работают без numpy/sklearn


In [8]:
# Test random_state reproducibility
print("=== Test: random_state reproducibility ===")

splitter1 = CUPACDataSplitter(n_folds=3, random_state=42, generate_cv_folds=True)
ml_data1 = splitter1.execute(exp_data)
ml_obj1 = ml_data1.get_ml_data('y')

splitter2 = CUPACDataSplitter(n_folds=3, random_state=42, generate_cv_folds=True)
ml_data2 = splitter2.execute(exp_data)
ml_obj2 = ml_data2.get_ml_data('y')

# Check if fold 0 has same indices
fold1_indices = ml_obj1.crossval[0][0].data.index.tolist()
fold2_indices = ml_obj2.crossval[0][0].data.index.tolist()

print(f"Fold 0 indices equal: {fold1_indices == fold2_indices}")
print(f"First 5 indices from splitter1: {fold1_indices[:5]}")
print(f"First 5 indices from splitter2: {fold2_indices[:5]}")

# Test with different random_state
splitter3 = CUPACDataSplitter(n_folds=3, random_state=123, generate_cv_folds=True)
ml_data3 = splitter3.execute(exp_data)
ml_obj3 = ml_data3.get_ml_data('y')

fold3_indices = ml_obj3.crossval[0][0].data.index.tolist()
print(f"\nFold 0 indices different with different random_state: {fold1_indices != fold3_indices}")
print(f"First 5 indices from splitter3: {fold3_indices[:5]}")

print("\n✓ random_state works correctly!")

=== Test: random_state reproducibility ===
Fold 0 indices equal: True
First 5 indices from splitter1: [83, 53, 70, 45, 44]
First 5 indices from splitter2: [83, 53, 70, 45, 44]

Fold 0 indices different with different random_state: True
First 5 indices from splitter3: [8, 70, 82, 28, 63]

✓ random_state works correctly!


# Final Test: CV без numpy (только Dataset API)

In [9]:
# Reload modules with final version
import sys
if 'hypex' in sys.modules:
    modules_to_remove = [key for key in sys.modules.keys() if key.startswith('hypex')]
    for module in modules_to_remove:
        del sys.modules[module]

from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator
from hypex.splitters import CUPACDataSplitter
from hypex.dataset import ExperimentData

print("✓ Final version loaded")

# Generate test data
gen = DataGenerator(
    n_samples=100,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
        "d": TreatmentRole(),
        "y": TargetRole(cofounders=["X1", "X2"]),
        "y_lag1": PreTargetRole(parent="y", lag=1),
        "X1_lag1": FeatureRole(parent="X1", lag=1),
        "X2_lag1": FeatureRole(parent="X2", lag=1),
        "y_lag2": PreTargetRole(parent="y", lag=2),
        "X1_lag2": FeatureRole(parent="X1", lag=2),
        "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

exp_data = ExperimentData(data=data)

# Test with CV folds (3 folds)
print("\n=== Test 1: CV folds generation ===")
splitter = CUPACDataSplitter(n_folds=3, random_state=42, generate_cv_folds=True)
ml_data = splitter.execute(exp_data)
ml_obj = ml_data.get_ml_data('y')

print(f"Number of CV folds: {len(ml_obj.crossval)}")
print(f"Fold sizes: {[ml_obj.crossval[i][0].shape[0] for i in range(3)]}")

# Test reproducibility
print("\n=== Test 2: Reproducibility ===")
splitter2 = CUPACDataSplitter(n_folds=3, random_state=42, generate_cv_folds=True)
ml_data2 = splitter2.execute(exp_data)
ml_obj2 = ml_data2.get_ml_data('y')

fold1_indices = ml_obj.crossval[0][0].data.index.tolist()
fold2_indices = ml_obj2.crossval[0][0].data.index.tolist()
print(f"Same indices with same random_state: {fold1_indices == fold2_indices}")

# Test different random_state
splitter3 = CUPACDataSplitter(n_folds=3, random_state=999, generate_cv_folds=True)
ml_data3 = splitter3.execute(exp_data)
ml_obj3 = ml_data3.get_ml_data('y')

fold3_indices = ml_obj3.crossval[0][0].data.index.tolist()
print(f"Different indices with different random_state: {fold1_indices != fold3_indices}")

print("\n✅ All tests passed! CV folds полностью работают через Dataset API (без numpy/sklearn)")

✓ Final version loaded

=== Test 1: CV folds generation ===
Number of CV folds: 3
Fold sizes: [34, 33, 33]

=== Test 2: Reproducibility ===
Same indices with same random_state: True
Different indices with different random_state: True

✅ All tests passed! CV folds полностью работают через Dataset API (без numpy/sklearn)


## Резюме изменений

### 1. **Создан базовый класс Splitter** (`hypex/splitters/base.py`)
   - Наследуется от `Executor`
   - Параметры: `n_folds`, `random_state`
   - Методы:
     - `create_cv_folds()` - **генерирует CV фолды используя только Dataset API** (без numpy/sklearn)
     - `add_cv_folds_to_mldata()` - добавляет CV фолды в MLData
     - `aggregate_columns()` - агрегирует временные колонки
     - `ensure_ml_experiment_data()` - преобразует ExperimentData → MLExperimentData

### 2. **Рефакторинг CUPACDataSplitter** (`hypex/splitters/cupac.py`)
   - Теперь наследуется от `Splitter` (вместо `Executor`)
   - Добавлен параметр `generate_cv_folds: bool = False`
   - Использует `self.aggregate_columns()` вместо `_agg_data_from_splits()`
   - Использует `self.add_cv_folds_to_mldata()` для генерации CV фолдов

### 3. **Создан класс MLData** (`hypex/dataset/ml_data.py`)
   - Поля:
     - `X_train: Dataset` - тренировочные признаки
     - `Y_train: Dataset` - тренировочные таргеты
     - `X_predict: Dataset` - данные для предсказания
     - `crossval: Dict[int, Tuple[Dataset, Dataset]]` - CV фолды

### 4. **Обновлен MLExperimentData** (`hypex/dataset/ml_data.py`)
   - `ml: Dict[str, MLData]` - словарь MLData объектов по таргетам
   - `trained_models`, `model_stats`, `fitted_transformers` - на верхнем уровне

### 5. **CV fold generation - 100% Dataset API**
   - Использует `Dataset.sample(frac=1.0, random_state=...)` для шафла
   - Использует `Dataset.iloc[indices]` для нарезки фолдов
   - **Никаких внешних зависимостей (numpy, sklearn)!**

### Все тесты проходят ✅

# Test: CUPAC без numpy

In [10]:
# Reload all modules to get updated CUPAC
import sys
if 'hypex' in sys.modules:
    modules_to_remove = [key for key in sys.modules.keys() if key.startswith('hypex')]
    for module in modules_to_remove:
        del sys.modules[module]

print("✓ Modules reloaded")

# Re-import and run baseline CUPAC test
from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator

# Generate test data
gen = DataGenerator(
    n_samples=100,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
        "d": TreatmentRole(),
        "y": TargetRole(cofounders=["X1", "X2"]),
        "y_lag1": PreTargetRole(parent="y", lag=1),
        "X1_lag1": FeatureRole(parent="X1", lag=1),
        "X2_lag1": FeatureRole(parent="X2", lag=1),
        "y_lag2": PreTargetRole(parent="y", lag=2),
        "X1_lag2": FeatureRole(parent="X1", lag=2),
        "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

print("\n=== Running CUPAC test with Dataset API only ===")
test_cupac_linear = ABTest(enable_cupac=True)
result_cupac_linear = test_cupac_linear.execute(data)
print("✅ CUPAC test completed successfully (no numpy!)")
result_cupac_linear.resume

✓ Modules reloaded

=== Running CUPAC test with Dataset API only ===
✅ CUPAC test completed successfully (no numpy!)


,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,y,1,1.015501,0.566974,-0.448527,-44.168021,NOT OK,0.676609
1,y_cupac,1,1.101453,0.400126,-0.701327,-63.672894,NOT OK,0.266649
